# 01 — Data Audit & Exploratory Data Analysis

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Objective

This notebook performs the **first and most important audit** of the raw NIFTY 50 stock dataset.

Before cleaning, feature engineering, portfolio construction, or machine learning, we need to understand:

- What data do we actually have?
- How large is the dataset?
- Which stocks and sectors are present?
- What time period is covered?
- Are there duplicate records?
- Where are the missing values?
- Do all stocks have the same historical coverage?
- Are the supplied features genuinely historical?
- Are there signs of survivorship bias?
- Are there possible look-ahead/leakage problems?
- What limitations must be documented?

### Important rule

**Never modify the raw dataset in this notebook.**

The raw CSV is our source of truth. Cleaning is handled separately in:

`02_price_integrity_and_cleaning.ipynb`


## 1. Project Research Question

Our final system will answer:

> **Given everything known about a portfolio today, what is the probability that it will experience a significant drawdown over the next 10 trading days?**

The initial target we plan to investigate is:

**1 = portfolio experiences a drawdown of at least 5% during the next 10 trading days**

**0 = otherwise**

We will later test 3%, 5%, and 10% thresholds before selecting the final target.

### Why this notebook comes first

Financial ML is highly sensitive to:

- bad prices
- missing observations
- survivorship bias
- look-ahead bias
- incorrect time ordering
- non-point-in-time fundamentals

Therefore, the dataset must be audited before any ML model is trained.


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RAW_PATH = Path("../data/raw/nifty50_historical_data.csv")

df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


## 2. Basic Dataset Overview

We first inspect the number of:

- observations (rows)
- variables (columns)
- unique stocks
- sectors
- dates

This gives us the basic dimensions of the problem.


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique tickers:", df["Ticker"].nunique())
print("Unique sectors:", df["Sector"].nunique())
print("Minimum date:", df["Date"].min())
print("Maximum date:", df["Date"].max())

print("\nColumn names:")
for col in df.columns:
    print("-", col)


## 3. Data Types

Correct data types are important.

For example:

- `Date` should eventually be a datetime
- OHLCV columns should be numeric
- ticker/company/sector should be categorical or string-like

We inspect the current types before changing anything.


In [ ]:
df.info()


## 4. Convert Date for Analysis

This conversion is performed only on the notebook copy.

The original CSV remains unchanged.


In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Invalid dates:", df["Date"].isna().sum())
print("Date range:", df["Date"].min(), "to", df["Date"].max())


## 5. Duplicate Records

We check two forms of duplication:

### Full duplicate
Exactly the same row appears more than once.

### Date + Ticker duplicate
The same stock appears more than once on the same date.

The second check is particularly important for daily stock data because we generally expect one observation per stock per trading date.


In [ ]:
full_duplicates = df.duplicated().sum()
date_ticker_duplicates = df.duplicated(["Date", "Ticker"]).sum()

print("Full duplicate rows:", full_duplicates)
print("Duplicate Date + Ticker rows:", date_ticker_duplicates)


## 6. Missing-Value Audit

Missing values are not automatically errors.

For example, a 200-day moving average cannot exist during the first ~200 observations of a stock's history.

We therefore report missingness first and decide how to handle each feature later.


In [ ]:
missing = (
    df.isna()
      .sum()
      .to_frame("Missing_Count")
)

missing["Missing_Percent"] = 100 * missing["Missing_Count"] / len(df)

missing = missing.sort_values("Missing_Count", ascending=False)

missing


## 7. Unique Values per Feature

This is an important leakage audit.

A feature such as historical market capitalization should normally change over time.

If a feature has approximately one value per ticker across 25 years, it may actually be a **current/static snapshot** rather than historical information.

We therefore inspect the number of unique values for every column and specifically investigate suspiciously static features.


In [ ]:
unique_counts = (
    df.nunique(dropna=False)
      .sort_values()
      .to_frame("Unique_Values")
)

unique_counts


## 8. Stock Universe

Let's inspect the actual stocks contained in the file.

The dataset is called NIFTY 50, but we must trust the actual contents of the CSV rather than the filename or description.


In [ ]:
stocks = (
    df.groupby("Ticker")
      .agg(
          Company_Name=("Company_Name", "first"),
          Sector=("Sector", "first"),
          First_Date=("Date", "min"),
          Last_Date=("Date", "max"),
          Observations=("Date", "size")
      )
      .sort_values("First_Date")
)

print("Number of stocks:", len(stocks))
stocks


## 9. Historical Coverage by Stock

NIFTY constituents do not all have the same historical start date.

Some stocks have data going back to 1999, while others only appear much later.

This matters because we must **not pretend that every current constituent existed in the index for the entire 1999–2026 period**.

This is one reason the project will explicitly document a **survivorship-bias limitation**.


In [ ]:
coverage = stocks.copy()

coverage["Years_of_Data"] = (
    coverage["Last_Date"] - coverage["First_Date"]
).dt.days / 365.25

coverage.sort_values("First_Date").head(20)


In [ ]:
print("Earliest-starting stocks:")
display(coverage.sort_values("First_Date").head(15))

print("\nLatest-starting stocks:")
display(coverage.sort_values("First_Date", ascending=False).head(15))


## 10. Observations per Stock

If every stock had exactly the same number of observations, their histories would be identical.

Different observation counts confirm that historical coverage varies across companies.


In [ ]:
coverage["Observations"].describe()


In [ ]:
plt.figure(figsize=(12, 6))
coverage["Observations"].sort_values().plot(kind="bar")
plt.title("Number of Historical Observations by Stock")
plt.xlabel("Stock")
plt.ylabel("Observations")
plt.xticks([])
plt.tight_layout()
plt.show()


## 11. Sector Distribution

Sector exposure matters because our eventual portfolio-risk system will include sector concentration and sector-level risk.

We inspect both:

1. number of stocks per sector
2. number of observations per sector


In [ ]:
sector_stock_count = (
    df.groupby("Sector")["Ticker"]
      .nunique()
      .sort_values(ascending=False)
)

sector_stock_count


In [ ]:
plt.figure(figsize=(10, 6))
sector_stock_count.sort_values().plot(kind="barh")
plt.title("Number of Stocks by Sector")
plt.xlabel("Number of Stocks")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()


## 12. Date-Level Coverage

We inspect how many stocks are available on each date.

This is important for portfolio construction because an equal-weight portfolio cannot simply assume that all 49 stocks are available on every historical date.


In [ ]:
daily_coverage = (
    df.groupby("Date")["Ticker"]
      .nunique()
)

daily_coverage.describe()


In [ ]:
plt.figure(figsize=(12, 5))
daily_coverage.plot()
plt.title("Number of Available Stocks Through Time")
plt.xlabel("Date")
plt.ylabel("Number of Stocks")
plt.tight_layout()
plt.show()


## 13. Price Sanity Checks

Before calculating returns, we inspect basic OHLC relationships.

Expected relationship:

- Low ≤ Open
- Low ≤ Close
- High ≥ Open
- High ≥ Close
- Low ≤ High
- Core prices should be positive

We do not clean anything here. We only identify potential problems.

Detailed cleaning is handled in Notebook 02.


In [ ]:
ohlc_invalid = (
    (df["Low"] > df["Open"]) |
    (df["Low"] > df["Close"]) |
    (df["High"] < df["Open"]) |
    (df["High"] < df["Close"]) |
    (df["Low"] > df["High"])
)

nonpositive_prices = (df[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)

print("Invalid OHLC rows:", ohlc_invalid.sum())
print("Rows with non-positive OHLC prices:", nonpositive_prices.sum())


## 14. Provided Daily Return vs Recalculated Return

The dataset contains a `Daily_Return` column.

We should not automatically trust it.

We compare it against:

\[
R_t = \frac{Close_t}{Close_{t-1}} - 1
\]

This helps determine whether the supplied return is simply derived from `Close`.


In [ ]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

df["Calculated_Return"] = df.groupby("Ticker")["Close"].pct_change()

comparison = df[["Daily_Return", "Calculated_Return"]].dropna()

comparison["Absolute_Difference"] = (
    comparison["Daily_Return"] - comparison["Calculated_Return"]
).abs()

print("Maximum absolute difference:",
      comparison["Absolute_Difference"].max())

print("Mean absolute difference:",
      comparison["Absolute_Difference"].mean())


## 15. Extreme Return Investigation

Large returns are not automatically errors.

A stock can legitimately move 20% or more because of:

- major news
- earnings
- regulatory events
- mergers
- corporate actions
- market stress

Therefore, extreme observations are **flagged for investigation**, not blindly deleted.

A separate price-integrity notebook performs the actual conservative cleaning.


In [ ]:
extreme_returns = df.loc[
    df["Calculated_Return"].abs() > 0.30,
    ["Date", "Ticker", "Company_Name", "Close",
     "Calculated_Return", "Stock_Split", "Dividend"]
].sort_values("Calculated_Return", key=lambda s: s.abs(), ascending=False)

print("Rows with absolute return > 30%:", len(extreme_returns))
extreme_returns.head(30)


## 16. Corporate-Action Fields

The dataset contains:

- `Stock_Split`
- `Dividend`

These fields are important when investigating unusually large price movements.

A stock split can make the raw price appear to fall sharply even though the investor's economic value has not fallen proportionally.

Therefore, suspicious price movements must be checked against corporate actions before being treated as genuine market crashes.


In [ ]:
print("Non-null Stock Split records:", df["Stock_Split"].notna().sum())
print("Non-null Dividend records:", df["Dividend"].notna().sum())

print("\nStock split examples:")
display(
    df.loc[df["Stock_Split"].notna(),
           ["Date", "Ticker", "Close", "Stock_Split"]]
      .head(20)
)


## 17. Fundamental-Feature Leakage Audit

The dataset includes fields such as:

- Market Cap
- PE Ratio
- Forward PE
- PEG Ratio
- Price-to-Book
- EPS
- Beta
- Dividend Yield
- 52-Week High
- 52-Week Low

These look useful, but we must ask:

> **Was this exact value actually known on the historical date represented by the row?**

If a stock has one constant value for a feature across many years, that is a strong warning that the value is a current/static snapshot.

Using such a feature to predict historical risk would introduce look-ahead bias.


In [ ]:
fundamental_cols = [
    "Market_Cap",
    "PE_Ratio",
    "Forward_PE",
    "PEG_Ratio",
    "Price_to_Book",
    "EPS",
    "Beta",
    "Dividend_Yield",
    "52Week_High",
    "52Week_Low"
]

fundamental_unique = (
    df.groupby("Ticker")[fundamental_cols]
      .nunique(dropna=False)
)

fundamental_unique


## 18. Identify Suspiciously Static Features

We calculate how many tickers have only one unique value for each fundamental field.

This is not proof by itself, but it is a strong diagnostic signal.


In [ ]:
static_feature_summary = pd.DataFrame({
    "Feature": fundamental_cols,
    "Tickers_with_1_unique_value": [
        (fundamental_unique[col] == 1).sum()
        for col in fundamental_cols
    ],
    "Total_Tickers": df["Ticker"].nunique()
})

static_feature_summary["Percent_of_Tickers"] = (
    100 * static_feature_summary["Tickers_with_1_unique_value"]
    / static_feature_summary["Total_Tickers"]
)

static_feature_summary.sort_values(
    "Percent_of_Tickers", ascending=False
)


## 19. Survivorship-Bias Assessment

### What is survivorship bias?

Suppose we look back to 2000 but use today's NIFTY 50 constituents.

Companies that failed, were removed from the index, merged, or were replaced may be absent.

That makes the historical universe look healthier than the real historical index universe.

### Our decision

We will **not claim** that this dataset represents the actual NIFTY 50 portfolio continuously from 1999.

Instead, we will clearly state:

> The project uses the stocks present in the supplied dataset/current constituent universe and their available historical observations. Historical constituent membership is not reconstructed.

This limitation will be documented in the final README and presentation.


## 20. Look-Ahead Bias Checklist

For every feature we eventually create, we will ask:

### Could this information have been known at the prediction date?

Allowed:

- past returns
- past volatility
- past prices
- past volume
- past drawdowns
- rolling correlations
- rolling beta calculated only from past data
- portfolio concentration calculated using current portfolio weights

Not allowed:

- future returns
- future volatility
- future drawdown
- future prices
- revised/current fundamentals inserted into historical rows
- any feature calculated using observations after the prediction date

This rule will govern the entire ML pipeline.


## 21. Preliminary Feature Decision

Based on this audit, the initial plan is:

### Raw data we keep

- Date
- Ticker
- Company_Name
- Sector
- Open
- High
- Low
- Close
- Volume
- Stock_Split
- Dividend

### Features we will recalculate ourselves

- Daily return
- Rolling volatility
- Moving averages
- Momentum
- Drawdown
- Rolling beta
- Rolling correlations
- Volume-based indicators

### Features requiring exclusion from the first model

The supplied static/current-style fundamental features will not be used as historical predictors unless we can establish that they are genuine point-in-time data.

This includes suspicious fields such as:

- Market Cap
- Forward PE
- EPS
- Price-to-Book
- 52-Week High
- 52-Week Low
- and other static/current-style fundamentals

`PEG_Ratio` is also unusable because it is completely missing.


## 22. Final Audit Summary

The output of this notebook is an **understanding of the raw dataset**, not a cleaned modeling dataset.

The next notebook:

`02_price_integrity_and_cleaning.ipynb`

will perform the actual conservative data-quality processing.

### Expected pipeline

**01 Data Audit & EDA**
→ Understand the raw data

**02 Price Integrity & Cleaning**
→ Flag and remove confirmed bad observations

**03 Stock Feature Engineering**
→ Build point-in-time stock-level features

**04 Portfolio Construction**
→ Build the portfolio

**05 Risk Target Creation**
→ Define future drawdown events

**06 Portfolio Feature Engineering**
→ Aggregate stock information into portfolio risk signals

**07–11 ML + Explainability**
→ Train, compare, calibrate, explain, and evaluate models


# Key Findings to Carry Forward

1. The file contains **49 stocks**, not 50.
2. The historical period is approximately **1999 to January 2026**.
3. Historical coverage differs significantly by stock.
4. There are **no duplicate Date + Ticker records** in the raw dataset.
5. Missing values exist and must be handled feature-by-feature.
6. Several supplied fundamental fields appear static/current rather than point-in-time.
7. `PEG_Ratio` is completely missing.
8. The supplied `Daily_Return` is derived from the price series and will be recalculated.
9. Large price/return anomalies require corporate-action and price-integrity checks.
10. The dataset has a **survivorship-bias limitation** that will be explicitly documented.
11. All future features must obey strict **point-in-time / no-look-ahead** rules.
